# Code Metrics — Style Transfer Project

Counts **files**, **classes**, **methods**, and **lines of code** for every Python file in the project,
then aggregates them into five categories: `core`, `ml`, `ui`, `tests`, and `ETC`.

## 1 — Import Required Libraries

In [ ]:
import ast
import os
from pathlib import Path
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("Libraries loaded.")

## 2 — Define Category Mappings

Files are assigned to a category by matching their path against the patterns below (first match wins).
Anything that doesn't match a known category lands in **ETC**.

In [ ]:
# Root of the project (one level up from docs/)
PROJECT_ROOT: Path = Path("..").resolve()

# Ordered list of (path-fragment, category-label).
# First matching rule wins; unmatched files → "ETC".
CATEGORY_RULES: list[tuple[str, str]] = [
    ("src/core",    "core"),
    ("src/ml",      "ml"),
    ("src/ui",      "ui"),
    ("tests",       "tests"),
]
CATEGORY_ORDER: list[str] = ["core", "ml", "ui", "tests", "ETC"]

# Directories to skip entirely
SKIP_DIRS: frozenset[str] = frozenset({".venv", "__pycache__", ".git", ".pytest_cache", ".mypy_cache"})

def assign_category(rel_path: Path) -> str:
    """Return the category label for a file given its path relative to PROJECT_ROOT."""
    parts = rel_path.as_posix()
    for fragment, label in CATEGORY_RULES:
        if parts.startswith(fragment):
            return label
    return "ETC"

print(f"Project root : {PROJECT_ROOT}")
print(f"Category rules: {[r[1] for r in CATEGORY_RULES]} + ETC")

## 3 — Scan Project Files and Collect Metrics

Walk the tree, parse every `.py` file with `ast`, and record per-file metrics.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class FileMetrics:
    rel_path: str
    category: str
    lines: int
    classes: int
    methods: int


def _count_metrics(py_file: Path) -> tuple[int, int, int]:
    """Return (lines, classes, methods) for a single Python file.

    Falls back to (line_count, 0, 0) if the file cannot be parsed.
    """
    source = py_file.read_text(encoding="utf-8", errors="replace")
    lines = len(source.splitlines())
    try:
        tree = ast.parse(source, filename=str(py_file))
    except SyntaxError:
        return lines, 0, 0

    classes = sum(1 for node in ast.walk(tree) if isinstance(node, ast.ClassDef))
    methods = sum(
        1
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    )
    return lines, classes, methods


def scan_project(root: Path) -> list[FileMetrics]:
    """Walk *root* and return one FileMetrics per .py file found."""
    records: list[FileMetrics] = []
    for dirpath, dirnames, filenames in os.walk(root):
        # Prune skip dirs in-place so os.walk won't descend into them
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for fname in sorted(filenames):
            if not fname.endswith(".py"):
                continue
            abs_path = Path(dirpath) / fname
            rel_path = abs_path.relative_to(root)
            category = assign_category(rel_path)
            lines, classes, methods = _count_metrics(abs_path)
            records.append(FileMetrics(
                rel_path=rel_path.as_posix(),
                category=category,
                lines=lines,
                classes=classes,
                methods=methods,
            ))
    return records


file_metrics: list[FileMetrics] = scan_project(PROJECT_ROOT)
print(f"Scanned {len(file_metrics)} Python files.")
for fm in file_metrics:
    print(f"  {fm.category:<8}  {fm.lines:>4} lines  {fm.classes:>2} cls  {fm.methods:>3} fn  {fm.rel_path}")

## 4 — Aggregate Metrics by Category

In [ ]:
df_files = pd.DataFrame(
    {
        "category": fm.category,
        "file":     fm.rel_path,
        "lines":    fm.lines,
        "classes":  fm.classes,
        "methods":  fm.methods,
    }
    for fm in file_metrics
)

# Aggregate: files = row count, others = sum
agg = (
    df_files
    .groupby("category", as_index=False)
    .agg(
        files=("file",    "count"),
        lines=("lines",   "sum"),
        classes=("classes", "sum"),
        methods=("methods", "sum"),
    )
)

# Ensure all categories appear even if empty, then sort by canonical order
agg = agg.set_index("category").reindex(CATEGORY_ORDER, fill_value=0).reset_index()
agg.rename(columns={"category": "Category", "files": "Files",
                     "lines": "Lines", "classes": "Classes",
                     "methods": "Methods"}, inplace=True)
print(agg)

## 5 — Metrics Table

In [ ]:
totals = pd.DataFrame([{
    "Category": "TOTAL",
    "Files":   agg["Files"].sum(),
    "Lines":   agg["Lines"].sum(),
    "Classes": agg["Classes"].sum(),
    "Methods": agg["Methods"].sum(),
}])

display_df = pd.concat([agg, totals], ignore_index=True)

# Style: right-align numeric cols, bold the TOTAL row
styled = (
    display_df.style
    .set_caption("Code Metrics by Category")
    .format({"Files": "{:,}", "Lines": "{:,}", "Classes": "{:,}", "Methods": "{:,"})
    .set_properties(**{"text-align": "right"}, subset=["Files", "Lines", "Classes", "Methods"])
    .set_properties(**{"text-align": "left"},  subset=["Category"])
    .apply(
        lambda row: ["font-weight: bold; background-color: #f0f0f0"] * len(row)
                    if row["Category"] == "TOTAL" else [""] * len(row),
        axis=1,
    )
    .hide(axis="index")
)

display(styled)

## 6 — Stacked Bar Chart

Four sub-charts — one per metric — each showing the per-category breakdown as a stacked bar.

In [ ]:
METRICS: list[str] = ["Files", "Lines", "Classes", "Methods"]
PALETTE: list[str] = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]

# Use only the category rows (drop TOTAL)
plot_df = agg.set_index("Category")

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Code Metrics by Category", fontsize=14, fontweight="bold", y=1.02)

for ax, metric in zip(axes, METRICS):
    values = plot_df[metric].values
    categories = plot_df.index.tolist()
    bottom = 0.0
    for i, (cat, val) in enumerate(zip(categories, values)):
        ax.bar(
            metric, val,
            bottom=bottom,
            color=PALETTE[i % len(PALETTE)],
            label=cat,
            edgecolor="white",
            linewidth=0.5,
        )
        if val > 0:
            ax.text(
                0, bottom + val / 2,
                f"{cat}\n{val:,}",
                ha="center", va="center",
                fontsize=8, color="white", fontweight="bold",
            )
        bottom += val

    ax.set_title(metric, fontsize=11)
    ax.set_xticks([])
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.set_ylabel(metric)
    ax.spines[["top", "right"]].set_visible(False)

# Single shared legend (built from the last axes to avoid duplicates)
handles, labels = axes[-1].get_legend_handles_labels()
# Deduplicate while preserving order
seen: dict[str, Any] = {}
for h, l in zip(handles, labels):
    if l not in seen:
        seen[l] = h
fig.legend(
    seen.values(), seen.keys(),
    loc="lower center", ncol=len(CATEGORY_ORDER),
    frameon=False, fontsize=9,
    bbox_to_anchor=(0.5, -0.05),
)

plt.tight_layout()
plt.savefig("code_metrics_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to docs/code_metrics_chart.png")